In [2]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

In [4]:
df=pd.read_csv("heart.csv")
df.head()

,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,target
0,63,1,3,145,233,1,0,150,0,2.3,0,0,1,1
1,37,1,2,130,250,0,1,187,0,3.5,0,0,2,1
2,41,0,1,130,204,0,0,172,0,1.4,2,0,2,1
3,56,1,1,120,236,0,1,178,0,0.8,2,0,2,1
4,57,0,0,120,354,0,1,163,1,0.6,2,0,2,1


In [5]:
df.shape

(303, 14)

In [64]:
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import RandomizedSearchCV

In [65]:
X=df.iloc[:,0:-1]
y=df.iloc[:,-1]

In [32]:
print(type(X), X.shape)
print(type(y), y.shape)

<class 'pandas.core.frame.DataFrame'> (303, 13)
<class 'pandas.core.series.Series'> (303,)


In [33]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42)

In [35]:
rf=RandomForestClassifier()
rf.fit(X_train,y_train)
y_pred=rf.predict(X_test)
print(accuracy_score(y_test,y_pred))

0.8524590163934426


In [40]:
rf=RandomForestClassifier(max_samples=0.75,random_state=42)
rf.fit(X_train,y_train)
y_pred=rf.predict(X_test)
print(accuracy_score(y_test,y_pred))

0.9016393442622951


### Cross Validation

In [47]:
from sklearn.model_selection import cross_val_score
x=np.mean(cross_val_score(RandomForestClassifier(max_samples=0.75),X,y,cv=10,scoring='accuracy'))
x

np.float64(0.8280645161290323)

## GridSearchCV

In [74]:
n_estimators=[20,60,100,120]
max_depth=[2,3,4]
max_features=[1,2,3]
max_samples=[0.5,0.75,1.0]
min_samples_split=[2,5]
bootstrap=[True,False]
min_samples_leaf = [1, 2]

In [56]:
param_grid={
     'n_estimators':n_estimators,
     'max_depth':max_depth,
     'max_features':max_features,
     'max_samples':max_samples
}
param_grid

{'n_estimators': [20, 60, 100, 120],
 'max_depth': [2, 3, 4],
 'max_features': [1, 2, 3],
 'max_samples': [0.5, 0.75, 1.0]}

In [57]:
rf_grid = GridSearchCV(estimator = rf, 
                       param_grid = param_grid, 
                       cv = 5, 
                       verbose=2, 
                       n_jobs = -1)

In [58]:
rf_grid.fit(X_train,y_train)

Fitting 5 folds for each of 108 candidates, totalling 540 fits


GridSearchCV(cv=5,
             estimator=RandomForestClassifier(max_samples=0.75,
                                              random_state=42),
             n_jobs=-1,
             param_grid={'max_depth': [2, 3, 4], 'max_features': [1, 2, 3],
                         'max_samples': [0.5, 0.75, 1.0],
                         'n_estimators': [20, 60, 100, 120]},
             verbose=2)

In [59]:
rf_grid.best_params_

{'max_depth': 2, 'max_features': 3, 'max_samples': 0.75, 'n_estimators': 100}

In [60]:
rf_grid.best_score_

np.float64(0.8428571428571429)

## Random Search CV

In [75]:
param_grid={
    'n_estimators':n_estimators,
    'max_depth':max_depth,
    'max_features':max_features,
    'max_samples':max_samples,
    'min_samples_split':min_samples_split,
    'bootstrap':bootstrap,
    'min_samples_leaf':min_samples_leaf 
}

In [76]:
rf_random=RandomizedSearchCV(estimator = rf, 
                       param_distributions = param_grid, 
                       cv = 5, 
                       verbose=2, 
                       n_jobs = -1)

In [77]:
rf_random.fit(X_train,y_train)

Fitting 5 folds for each of 10 candidates, totalling 50 fits


C:\Users\Priti\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\model_selection\_validation.py:528: FitFailedWarning: 
40 fits failed out of a total of 50.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
40 fits failed with the following error:
Traceback (most recent call last):
  File "C:\Users\Priti\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\model_selection\_validation.py", line 866, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
    ~~~~~~~~~~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "C:\Users\Priti\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\base.py", line 1389, in wrapper
    return fit_method(estimator, *args, **kwargs)
  File "C:\Users\Priti\AppDat

RandomizedSearchCV(cv=5,
                   estimator=RandomForestClassifier(max_samples=0.75,
                                                    random_state=42),
                   n_jobs=-1,
                   param_distributions={'bootstrap': [True, False],
                                        'max_depth': [2, 3, 4],
                                        'max_features': [1, 2, 3],
                                        'max_samples': [0.5, 0.75, 1.0],
                                        'min_samples_leaf': [1, 2],
                                        'min_samples_split': [2, 5],
                                        'n_estimators': [20, 60, 100, 120]},
                   verbose=2)

In [78]:
rf_random.best_params_

{'n_estimators': 120,
 'min_samples_split': 5,
 'min_samples_leaf': 1,
 'max_samples': 0.75,
 'max_features': 1,
 'max_depth': 2,
 'bootstrap': True}

In [80]:
rf_random.best_score_

np.float64(0.7930272108843538)

## OOB SCORE

In [81]:
rf=RandomForestClassifier(oob_score=True)

In [82]:
rf.fit(X_train,y_train)

RandomForestClassifier(oob_score=True)

In [83]:
rf.oob_score_         ## roughfly tells the accuracy of model

0.8140495867768595

In [86]:
y_pred=rf.predict(X_test)
accuracy_score(y_pred,y_test)

0.8524590163934426

## Feature importance

In [87]:
rf.feature_importances_

array([0.08683614, 0.03865478, 0.11996092, 0.07285207, 0.07472239,
       0.00887489, 0.02081088, 0.10686014, 0.0642762 , 0.13751426,
       0.0490025 , 0.12988915, 0.08974567])

ValueError: cannot reshape array of size 13 into shape (303,14)